# Executive Summary - Task 3: Filter Validation and Robustness

## Problem Statement

Given a BTC perp dataset (804M trades, ~180 calendar days). In Task 2, threshold filters were built on five features: VPIN (5s), distance-to-mid, Binance liquidations (30s), Bybit liquidations (30s), OBI. The goal of Task 3 is to statistically prove that these filters work, assess their robustness over time, build a monitoring system, and make a justified decision: accept, reject, or observe.

All calculations are performed on a chronological subsample (80M rows). Consistency with the full dataset is confirmed: the PnL_all deviation is less than 5 standard errors of the subsample.

---

## 1. Official Score and Sanity Checks

Score is implemented exactly per specification: `Score(τ) = PnL_kept(τ) − PnL_all(τ)` with weights `w_i = min(notional_i, 100,000)`, rebate +0.5 bps, and turnover constraint ≥ 500,000 USD/day. PnL_all from Task 2 is reproduced: −0.105 bps (30s), −0.037 bps (120s), +0.009 bps (300s).

Sanity checks:
- Random filter (30%, 100 realizations): Score ~= 0, σ ~= 0.005 bps.
- Oracle filter (pnl < 0): Score ≈ 1.1 bps (30s) - upper bound. All real filters are below.
- Turnover constraint is satisfied for all filters at current thresholds.

Best filters by Score(30s): Dist-to-mid (+0.55 bps), Liq Bybit (+0.42 bps), Liq Binance (+0.39 bps), VPIN (+0.04 bps), OBI (+0.07 bps).

---

## 2. Relationship Between ML Quality and Score

Each selected filter was treated as a binary classifier of "toxic" trades (pnl < 0). Weighted precision, recall, F1, and MCC were computed.

Key finding: the correlation between MCC/F1 and Score is positive but not perfect. Score depends not only on the correctness of the PnL sign (what the classifier predicts) but also on its magnitude — a filter that catches large losing trades accumulates more Score than a filter with higher recall but on small losses. This is a limitation of the "optimize the classifier → get the best Score" approach.

---

## 3. Robustness: Walk-Forward, Rolling Window, Purged CV, CPCV

Four validation schemes were applied, from the most realistic to the most informative in terms of variance:

**Walk-forward (30 days train / 15 days test, 9 folds)** - the primary realistic evaluation. All five filters (except OBI) are consistently positive across all folds. Random is consistently around zero. Purge and embargo were not applied: thresholds are computed from backward-looking features, markout leakage ≤ 0.3% of the last train day - negligible at daily granularity.

**Rolling window (14/7/7, ~24 folds)** — same conclusion, but more data points.

**Purged K-fold CV (K=5, purge=1d, embargo=1d)** - estimation of Score variance across regimes. In middle folds (1–3), the training set contains future data; purge and embargo protect against markout leakage but not against look-ahead in threshold statistics. This is a deliberate assumption: purged CV is used as an estimate of cross-temporal stability, not as a deployment simulation.

**CPCV (6 groups, n_test=2, C(6,2)=15 combinations)** - 15 combinations instead of 5, sufficient for constructing a confidence interval. All strong filters are positive across all 15 combinations.

---

## 4. Autocorrelation and Bootstrap Selection

ACF(1) of daily Scores for all filters lies within the Bartlett band (±1.96/√T). This is a key empirical result: daily aggregation completely eliminates autocorrelation caused by overlapping markout windows (τ ≤ 300s ≪ 86,400s). If the analysis were conducted at the per-trade level, IID bootstrap would catastrophically underestimate variance.

At the daily level, IID and block bootstrap produce nearly identical CI widths (ratio 0.88–1.15). This does not mean that block methods were redundant - it means that the choice of daily aggregation was methodologically correct. Six methods are demonstrated (IID, MBB, CBB, Stationary, Bayesian, Bayesian-Block).

---

## 5. How to Prove That Model A Is Better Than Model B?

**For two models:** Diebold-Mariano test with Newey-West (HAC) variance. The test uses the series of daily differences `d_t = Score_A(t) − Score_B(t)`, autocorrelation is accounted for by the Bartlett kernel with lag according to the Andrews rule.

DM-HAC results (H0: filter is no better than Random):

| Filter                    | DM stat. | p-value | 95% CI (bps/day)   |
|---|---|---|---|
| Dist-to-mid (q=10%)      | +6.25    | <0.001  | [+0.36, +0.70]     |
| Liq Binance 30s (q=10%)  | +7.90    | <0.001  | [+0.32, +0.53]     |
| Liq Bybit 30s (q=10%)    | +5.02    | <0.001  | [+0.31, +0.72]     |
| VPIN 5s (q=70%)          | +3.98    | <0.001  | [+0.02, +0.06]     |
| OBI (q=90%)              | +1.65    | 0.049   | [−0.01, +0.16]     |

Four out of five filters significantly outperform Random at p < 0.001. OBI is at the boundary of significance.

Confirmed by permutation test (sign-flip, 10,000 permutations): results match for four strong filters; OBI is not rejected (p = 0.049).

---

## 6. What If There Are N Models? Multiple Testing

Three methods in order of increasing power:

**Hansen SPA** (White's Reality Check with stationary bootstrap). H0: "no filter is better than Random." Test statistic +7.90, p < 0.001. H0 is rejected - at least one filter outperforms Random accounting for data snooping.

**Model Confidence Set (MCS)** - iterative elimination of the worst model until the remaining ones are indistinguishable. The MCS includes three filters: Dist-to-mid, Liq Binance, Liq Bybit. Eliminated: Random (p = 0), VPIN (p = 0), OBI (p = 0). Conclusion: we cannot select a single "best" among the trio of filters, but this trio is statistically separated from the rest.

**Romano-Wolf stepdown** (FWER control). All five filters are rejected with adj. p < 0.05. OBI is borderline (adj. p = 0.041), the rest adj. p < 0.001.

---

## 7. Leakage Audit (Look-Ahead Bias)

Comparison of full-sample thresholds and expanding-window thresholds (warmup = 60 days):

| Filter                 | Score (full) | Score (expanding) | delta (bps) | Deflation (%) | Verdict |
|---|---|---|---|---|---|
| VPIN 5s                | +0.045       | +0.045            | ~=0      | −0.04%        | stable  |
| Dist-to-mid            | +0.553       | +0.527            | +0.025  | +4.6%         | stable  |
| Liq Binance 30s        | +0.386       | +0.382            | +0.004  | +1.0%         | stable  |
| Liq Bybit 30s          | +0.423       | +0.430            | −0.007  | −1.7%         | stable  |

All filters received the verdict "stable" (deflation < 10%). Thresholds are stable over time - full-sample quantiles and expanding-window quantiles differ insignificantly. Look-ahead bias is not a problem for our baselines.

---

## 8. Monitoring System for Live Deployment

Four layers:

**8.1. Feature drift (PSI).** Population Stability Index for 11 features (train: first 120 days vs validation: remainder). Threshold: PSI < 0.10 - OK, 0.10–0.25 - investigate, > 0.25 - recalibrate. Visualization: PSI bar chart + train vs val histograms for the top 4 drifting features.

**8.2. Score drift (CUSUM).** Two-sided CUSUM with threshold H = 4sigma and slack K = 0.5sigma. On the best filter (Dist-to-mid): one positive alarm on day 50. No negative alarms - the filter is not degrading.

**8.3. Rolling Score with control bands.** 7-day rolling mean of Score with +-2sigma bands from train statistics. No breaches of either upper or lower bounds.

**8.4. Killswitch.** Automated stop rules: Score < 0 for seven consecutive days -> stop; PSI > 0.25 on a key feature -> recalibrate; turnover < 500k/day -> stop. Based on evaluation on the validation period, the killswitch was not triggered.

---
